In [1]:
# Use the shell escape (!) to run the script on Colab's cloud filesystem
!python -c "import torch; print('CUDA Available:', torch.cuda.is_available()); print('GPU Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"


CUDA Available: True
GPU Device: NVIDIA L4


In [2]:
!ls

sample_data


In [3]:
import os
from google.colab import drive

# 1. Mount Google Drive to access your processed dataset
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define repository details (Using public HTTPS url)
repo_name = "SME_Credit_Risk"
repo_url = f"https://github.com/mirkosimunovic/{repo_name}.git"

# 3. Clone the public repo (or pull if it already exists)
if not os.path.exists(f"/content/{repo_name}"):
    print(f"\nCloning public repository: {repo_url}...")
    !git clone {repo_url}
else:
    print(f"\nRepository {repo_name} already exists. Pulling latest code changes...")
    %cd /content/{repo_name}
    !git pull
    %cd /content

# 4. Create the target directory inside the cloned repo
!mkdir -p /content/{repo_name}/data/processed

# 5. Copy the cleaned SME dataset from Google Drive
# NOTE: If you saved the CSV inside a specific folder in Google Drive, 
# adjust the source path below (e.g., "/content/drive/MyDrive/YourFolder/processed_sme_final.csv")
drive_source_path = "/content/drive/MyDrive/xAI_Banking_Paper/data/processed/processed_sme_final.csv"
colab_target_path = f"/content/{repo_name}/data/processed/processed_sme_final.csv"

if os.path.exists(drive_source_path):
    !cp "{drive_source_path}" "{colab_target_path}"
    print("\n✓ Cleaned SBA dataset successfully copied from Google Drive to the cloned project")
else:
    print(f"\n⚠️ WARNING: Could not find your dataset at: {drive_source_path}")
    print("Please check your file path inside your Google Drive side panel and update 'drive_source_path'.")

# 6. Change active directory to your repository root
%cd /content/{repo_name}
print(f"\nActive directory set to: {os.getcwd()}")

Mounted at /content/drive

Cloning public repository: https://github.com/mirkosimunovic/SME_Credit_Risk.git...
Cloning into 'SME_Credit_Risk'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 104 (delta 31), reused 90 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 18.34 MiB | 20.42 MiB/s, done.
Resolving deltas: 100% (31/31), done.

✓ Cleaned SBA dataset successfully copied from Google Drive to the cloned project
/content/SME_Credit_Risk

Active directory set to: /content/SME_Credit_Risk


In [12]:
!git pull


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.33 KiB | 2.33 MiB/s, done.
From https://github.com/mirkosimunovic/SME_Credit_Risk
   a6fe197..e6bf224  main       -> origin/main
Updating a6fe197..e6bf224
Fast-forward
 scripts/trainer.py | 154 ++++++++++++++++++++++++++++++++++++++++++-----------
 1 file changed, 124 insertions(+), 30 deletions(-)


In [5]:

# 1. Install heavy, CUDA-dependent system libraries first
!pip install "fknni[rapids12]" --extra-index-url=https://pypi.nvidia.com
!pip install faiss-gpu-cu12

# 2. Silently install the rest of our standard project requirements
!pip install -q -r requirements.txt


Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 94.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 266.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.5/114.5 kB 150.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 78.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 GB 32.9 MB/s eta 0:00:00:00:0100:01
INFO: pip is looking at multiple versions of cugraph-cu12 to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 203.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 224.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 140.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 85.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3

In [10]:
!find . -not -path './.git/*' -not -path './__pycache__/*' | sed -e 's|[^/]*/|│   |g' -e 's|│   \([^│]\)|├── \1|'

.
├── .gitignore
├── outputs
│   ├── figures
│   │   ├── roc_curves_comparison.png
│   ├── results
│   │   ├── metrics_benchmark.csv
├── .git
├── .cursorrules
├── notebooks
│   ├── SME_week1.ipynb
│   ├── run_commands.ipynb
├── data
│   ├── processed
│   │   ├── processed_sme_final.csv
│   │   ├── y_train.csv
│   │   ├── X_train.csv
│   │   ├── y_oot.csv
│   │   ├── X_oot.csv
├── scripts
│   ├── trainer.py
│   ├── preprocess.py
├── models
│   ├── artifacts
│   │   ├── .gitkeep
├── requirements.txt


In [13]:
!python scripts/trainer.py

SME Credit Risk — two-step trainer + OOT evaluation
Project root: /content/SME_Credit_Risk

Creating output directories if missing:
  [ok] outputs/figures
  [ok] outputs/results
  [ok] models/artifacts

Loading chronological splits (NaNs still present):
  Loaded train: X=(852308, 23)  defaults=152,527  rate=0.178958
  Loaded oot: X=(44859, 23)  defaults=5,031  rate=0.112151

STEP 1  Stratified 5-fold CV on X_train / y_train
Imputer: FastKNNImputer.fit_transform on the train fold, then on stacked val.
         (Library has no fit/transform; val neighbors come from imputed train.)
Scaler:  StandardScaler on continuous columns (nunique > 10) — train fold only.
Imbalance: scale_pos_weight = n_neg / n_pos on the training fold.
OOT is held out of this entire loop.

Fold 1/5  n_train=681,846  n_val=170,462  scale_pos_weight=4.59
    FastKNNImputer.fit_transform on training slice (681,846 rows) ...
    FastKNNImputer.fit_transform on stacked reference+apply (852,308 rows) ...
    Continuous co

# Sync the Colab session file to my Google Drive drive

In [14]:
import shutil
from pathlib import Path

src = Path("/content/SME_Credit_Risk")
dst = Path("/content/drive/MyDrive/xAI_Banking_Paper/SME_Credit_Risk")  # change if your Drive folder differs

files = [
    "models/artifacts/imputer.joblib",
    "models/artifacts/scaler.joblib",
    "models/artifacts/xgboost_best.json",
    "models/artifacts/lightgbm_best.txt",
    "models/artifacts/catboost_best.bin",
    "outputs/results/metrics_benchmark.csv",
    "outputs/figures/roc_curves_comparison.png",
]
for rel in files:
    dest = dst / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src / rel, dest)
    print("copied", rel)

copied models/artifacts/imputer.joblib
copied models/artifacts/scaler.joblib
copied models/artifacts/xgboost_best.json
copied models/artifacts/lightgbm_best.txt
copied models/artifacts/catboost_best.bin
copied outputs/results/metrics_benchmark.csv
copied outputs/figures/roc_curves_comparison.png
